# Bernstein-Vazirani Algorithm

## Introduction

The **Bernstein-Vazirani algorithm**, published by Ethan Bernstein and Umesh Vazirani in 1993, was one of the first algorithms to demonstrate a provable quantum speedup over classical computation.

### The problem

We are given a black-box function (oracle)

    f : {0,1}^n  ->  {0,1}
    f(x) = s . x  (mod 2)

where `.` denotes the **bitwise inner product** (also called the dot product mod 2):

    s . x = (s_{n-1} * x_{n-1}) XOR ... XOR (s_1 * x_1) XOR (s_0 * x_0)

and `s` is an unknown `n`-bit secret string.  The goal is to **find `s`**.

### Notation

- $n$ -- number of bits in the secret string
- $s$ -- the hidden $n$-bit string we want to find
- $x$ -- an $n$-bit query input
- $s \cdot x$ -- bitwise inner product mod 2: $(s_0 x_0) \oplus \cdots \oplus (s_{n-1} x_{n-1})$
- $H^{\otimes n}$ -- $n$-fold tensor product of the Hadamard gate
- $U_f$ -- phase-kickback oracle for $f$
- $e_i$ -- unit vector with a 1 only at position $i$

### Numeric example  (n = 3, s = 101)

Let `s = [1, 0, 1]` and evaluate `f` on the three unit vectors:

    x = 100  ->  f(100) = 1*1 XOR 0*0 XOR 1*0 = 1   (reveals s_0 = 1)
    x = 010  ->  f(010) = 1*0 XOR 0*1 XOR 1*0 = 0   (reveals s_1 = 0)
    x = 001  ->  f(001) = 1*0 XOR 0*0 XOR 1*1 = 1   (reveals s_2 = 1)

Classically, three queries are needed to reconstruct `s = 101`.

The Bernstein-Vazirani circuit queries the oracle **once** with a
superposition of all 2^3 = 8 inputs simultaneously.  After the second
H^3 the query register collapses deterministically to `|101>` = `|s>`.

### Classical vs quantum complexity

Classically, to determine all `n` bits of `s` you need at least `n` queries to the oracle (one per bit, e.g. by setting `x = e_i`, the i-th unit vector).  
The Bernstein-Vazirani algorithm finds `s` with **a single query** to the quantum oracle.

### Circuit

```
                   +-----------------+
|x>  |0>-/n--H^n--|                 |--/n--H^n---Measure --> s
                  |       U_f       |
|y>  |1>---H------|                 |------------------------
                   +-----------------+
```

Steps:
1. Prepare `n` query qubits in `|0>^n` and one ancilla qubit in `|1>`.
2. Apply `H^n` to the query register and `H` to the ancilla -- this creates a uniform superposition for the query and the `|->` state for the ancilla.
3. Apply the phase-kickback oracle `U_f`: for each `|x>` component, the ancilla flips to `|y XOR f(x)>`.  Because the ancilla is `|->`, this is equivalent to multiplying the amplitude of `|x>` by `(-1)^(s.x)` (phase kickback).
4. Apply `H^n` to the query register again.
5. Measure the query register.  The result is `s` with probability 1.

### Imports

- **SymPy quantum** -- `qapply`, `Qubit`, `HadamardGate`, `TensorProduct`, `represent`, `measure_partial_oneshot`.
- `matrix_to_qubit` / `represent` -- convert between matrix and qubit representations.
- `preorder_traversal` -- walk the SymPy expression tree to extract the measured `Qubit`.
- `numpy` -- random bit-string generation for tests.
- `oracle` -- the phase-kickback quantum oracle from `oracle.py`.
- `hn` -- helper that builds an `n`-qubit Hadamard gate `H^n`.

In [1]:
"""Bernstein-Vazirani algorithm -- find a hidden bit string in one query."""

from sympy.physics.quantum.qapply import qapply
from sympy.physics.quantum.qubit import (
    Qubit,
    matrix_to_qubit,
    measure_partial_oneshot,
)
from sympy.physics.quantum import TensorProduct
from sympy.physics.quantum.gate import HadamardGate
from sympy.physics.quantum.represent import represent
from sympy import preorder_traversal
import numpy as np

from oracle import oracle
from util.util import hn

## The oracle function

The classical inner-product function `f(x) = s . x (mod 2)` is implemented here as `fbv`.  It computes the bitwise dot product using XOR instead of ordinary addition, which keeps every intermediate result in `{0, 1}`.

In [2]:
def fbv(x, s):
    """Compute the bitwise inner product f(x) = s . x  (mod 2).

    Each pair of corresponding bits is ANDed (s_i * x_i) and the results
    are combined with XOR, which is addition mod 2.

    :param x: iterable of bits representing the query input
    :param s: iterable of bits representing the secret string
    :return: 0 or 1 -- the inner product mod 2
    """
    result = 0
    for xi, si in zip(x, s):
        result ^= xi * si  # XOR accumulates (mod-2 sum)
    return result

## The Bernstein-Vazirani quantum circuit

The function below implements the full circuit:

1. **Query register** `|x>` -- start from `|0...0>` and apply `H^n` to produce a uniform superposition over all `n`-bit strings.
2. **Ancilla qubit** `|y>` -- start from `|1>` and apply `H` to produce `|-> = (|0> - |1>) / sqrt(2)`.
3. **Tensor product** -- combine the two registers into a single `(n+1)`-qubit state `|x> |y>`.
4. **Oracle** `U_f` -- applies phase kickback: `|x>|-> -> (-1)^(s.x) |x>|->`.
5. **Second `H^n`** on the query register -- interference collapses the superposition to exactly `|s>`.

In [3]:
def berstein_vazirani(s, n):
    """Run the Bernstein-Vazirani algorithm to recover the secret string s.

    The algorithm uses a single call to the phase-kickback oracle U_f
    where f(x) = s.x (mod 2), then recovers s exactly via H^n.

    :param s: iterable of n bits -- the secret string to recover
    :param n: length of the secret string (number of query qubits)
    :return: the (n+1)-qubit state |s>|ancilla> after the second H^n
    """
    # Step 1: query register |0>^n -> H^n -> uniform superposition
    x = qapply(hn(n) * Qubit("0" * n))
    print(f"|x>: {x}")

    # Step 2: ancilla |1> -> H -> |-> = (|0> - |1>) / sqrt(2)
    y = qapply(HadamardGate(0) * Qubit(1))
    print(f"|y>: {y}")

    # Step 3: combine into a single (n+1)-qubit register
    xy = TensorProduct(x, y)
    xy = matrix_to_qubit(represent(xy))
    print(f"|xy>: {xy}")

    # Step 4: apply the phase-kickback oracle U_f
    r = oracle(x, y, fbv, s)

    # Step 5: apply H^n to the query register (qubits 1..n, skipping ancilla 0)
    r = qapply(hn(n, start=1) * r)
    return r

## Tests

In [4]:
def _extract_s(m, n):
    """Extract the n-qubit secret string from a partial measurement result.

    The measurement result is an (n+1)-qubit state; the ancilla occupies
    the lowest qubit (index 0) and the query bits occupy indices 1..n.
    We walk the SymPy expression tree to find the Qubit and slice off
    the top n bits.

    :param m: SymPy expression returned by measure_partial_oneshot
    :param n: number of secret-string bits
    :return: Qubit containing the n measured bits
    """
    for arg in preorder_traversal(m):
        if isinstance(arg, Qubit):
            return Qubit(*arg.qubit_values[0:n])
    return None

In [5]:
def test_0():
    """Random 4-bit secret string: algorithm must recover it exactly."""
    size = 4
    s = np.random.choice([0, 1], size=(size,))
    print(f"\ntest_0: secret s = {s}")

    r = berstein_vazirani(s, size)
    print(f"Circuit output: {r}")

    m = measure_partial_oneshot(r, range(size))
    s_found = _extract_s(m, size)
    print(f"Recovered s    : {s_found}")

    assert Qubit(*s) == s_found


test_0()


test_0: secret s = [1 0 0 1]
|x>: |0000>/4 + |0001>/4 + |0010>/4 + |0011>/4 + |0100>/4 + |0101>/4 + |0110>/4 + |0111>/4 + |1000>/4 + |1001>/4 + |1010>/4 + |1011>/4 + |1100>/4 + |1101>/4 + |1110>/4 + |1111>/4
|y>: sqrt(2)*|0>/2 - sqrt(2)*|1>/2
|xy>: sqrt(2)*|00000>/8 - sqrt(2)*|00001>/8 + sqrt(2)*|00010>/8 - sqrt(2)*|00011>/8 + sqrt(2)*|00100>/8 - sqrt(2)*|00101>/8 + sqrt(2)*|00110>/8 - sqrt(2)*|00111>/8 + sqrt(2)*|01000>/8 - sqrt(2)*|01001>/8 + sqrt(2)*|01010>/8 - sqrt(2)*|01011>/8 + sqrt(2)*|01100>/8 - sqrt(2)*|01101>/8 + sqrt(2)*|01110>/8 - sqrt(2)*|01111>/8 + sqrt(2)*|10000>/8 - sqrt(2)*|10001>/8 + sqrt(2)*|10010>/8 - sqrt(2)*|10011>/8 + sqrt(2)*|10100>/8 - sqrt(2)*|10101>/8 + sqrt(2)*|10110>/8 - sqrt(2)*|10111>/8 + sqrt(2)*|11000>/8 - sqrt(2)*|11001>/8 + sqrt(2)*|11010>/8 - sqrt(2)*|11011>/8 + sqrt(2)*|11100>/8 - sqrt(2)*|11101>/8 + sqrt(2)*|11110>/8 - sqrt(2)*|11111>/8
Circuit output: sqrt(2)*|10010>/2 - sqrt(2)*|10011>/2
Recovered s    : |1001>


In [6]:
def test_1():
    """Fixed 4-bit secret string 1010: algorithm must return |1010>."""
    s = [1, 0, 1, 0]
    size = len(s)
    print(f"\ntest_1: secret s = {s}")

    r = berstein_vazirani(s, size)
    m = measure_partial_oneshot(r, range(size))
    s_found = _extract_s(m, size)
    print(f"Recovered s = {s_found}")

    assert Qubit(*s) == s_found


test_1()


test_1: secret s = [1, 0, 1, 0]
|x>: |0000>/4 + |0001>/4 + |0010>/4 + |0011>/4 + |0100>/4 + |0101>/4 + |0110>/4 + |0111>/4 + |1000>/4 + |1001>/4 + |1010>/4 + |1011>/4 + |1100>/4 + |1101>/4 + |1110>/4 + |1111>/4
|y>: sqrt(2)*|0>/2 - sqrt(2)*|1>/2
|xy>: sqrt(2)*|00000>/8 - sqrt(2)*|00001>/8 + sqrt(2)*|00010>/8 - sqrt(2)*|00011>/8 + sqrt(2)*|00100>/8 - sqrt(2)*|00101>/8 + sqrt(2)*|00110>/8 - sqrt(2)*|00111>/8 + sqrt(2)*|01000>/8 - sqrt(2)*|01001>/8 + sqrt(2)*|01010>/8 - sqrt(2)*|01011>/8 + sqrt(2)*|01100>/8 - sqrt(2)*|01101>/8 + sqrt(2)*|01110>/8 - sqrt(2)*|01111>/8 + sqrt(2)*|10000>/8 - sqrt(2)*|10001>/8 + sqrt(2)*|10010>/8 - sqrt(2)*|10011>/8 + sqrt(2)*|10100>/8 - sqrt(2)*|10101>/8 + sqrt(2)*|10110>/8 - sqrt(2)*|10111>/8 + sqrt(2)*|11000>/8 - sqrt(2)*|11001>/8 + sqrt(2)*|11010>/8 - sqrt(2)*|11011>/8 + sqrt(2)*|11100>/8 - sqrt(2)*|11101>/8 + sqrt(2)*|11110>/8 - sqrt(2)*|11111>/8
Recovered s = |1010>


In [7]:
def test_2():
    """All-zeros secret string: algorithm must return |0000>."""
    s = [0, 0, 0, 0]
    size = len(s)
    print(f"\ntest_2: secret s = {s}")

    r = berstein_vazirani(s, size)
    m = measure_partial_oneshot(r, range(size))
    s_found = _extract_s(m, size)
    print(f"Recovered s = {s_found}")

    assert Qubit(*s) == s_found


test_2()


test_2: secret s = [0, 0, 0, 0]
|x>: |0000>/4 + |0001>/4 + |0010>/4 + |0011>/4 + |0100>/4 + |0101>/4 + |0110>/4 + |0111>/4 + |1000>/4 + |1001>/4 + |1010>/4 + |1011>/4 + |1100>/4 + |1101>/4 + |1110>/4 + |1111>/4
|y>: sqrt(2)*|0>/2 - sqrt(2)*|1>/2
|xy>: sqrt(2)*|00000>/8 - sqrt(2)*|00001>/8 + sqrt(2)*|00010>/8 - sqrt(2)*|00011>/8 + sqrt(2)*|00100>/8 - sqrt(2)*|00101>/8 + sqrt(2)*|00110>/8 - sqrt(2)*|00111>/8 + sqrt(2)*|01000>/8 - sqrt(2)*|01001>/8 + sqrt(2)*|01010>/8 - sqrt(2)*|01011>/8 + sqrt(2)*|01100>/8 - sqrt(2)*|01101>/8 + sqrt(2)*|01110>/8 - sqrt(2)*|01111>/8 + sqrt(2)*|10000>/8 - sqrt(2)*|10001>/8 + sqrt(2)*|10010>/8 - sqrt(2)*|10011>/8 + sqrt(2)*|10100>/8 - sqrt(2)*|10101>/8 + sqrt(2)*|10110>/8 - sqrt(2)*|10111>/8 + sqrt(2)*|11000>/8 - sqrt(2)*|11001>/8 + sqrt(2)*|11010>/8 - sqrt(2)*|11011>/8 + sqrt(2)*|11100>/8 - sqrt(2)*|11101>/8 + sqrt(2)*|11110>/8 - sqrt(2)*|11111>/8
Recovered s = |0000>


In [8]:
def test_3():
    """All-ones secret string: algorithm must return |1111>."""
    s = [1, 1, 1, 1]
    size = len(s)
    print(f"\ntest_3: secret s = {s}")

    r = berstein_vazirani(s, size)
    m = measure_partial_oneshot(r, range(size))
    s_found = _extract_s(m, size)
    print(f"Recovered s = {s_found}")

    assert Qubit(*s) == s_found


test_3()


test_3: secret s = [1, 1, 1, 1]
|x>: |0000>/4 + |0001>/4 + |0010>/4 + |0011>/4 + |0100>/4 + |0101>/4 + |0110>/4 + |0111>/4 + |1000>/4 + |1001>/4 + |1010>/4 + |1011>/4 + |1100>/4 + |1101>/4 + |1110>/4 + |1111>/4
|y>: sqrt(2)*|0>/2 - sqrt(2)*|1>/2
|xy>: sqrt(2)*|00000>/8 - sqrt(2)*|00001>/8 + sqrt(2)*|00010>/8 - sqrt(2)*|00011>/8 + sqrt(2)*|00100>/8 - sqrt(2)*|00101>/8 + sqrt(2)*|00110>/8 - sqrt(2)*|00111>/8 + sqrt(2)*|01000>/8 - sqrt(2)*|01001>/8 + sqrt(2)*|01010>/8 - sqrt(2)*|01011>/8 + sqrt(2)*|01100>/8 - sqrt(2)*|01101>/8 + sqrt(2)*|01110>/8 - sqrt(2)*|01111>/8 + sqrt(2)*|10000>/8 - sqrt(2)*|10001>/8 + sqrt(2)*|10010>/8 - sqrt(2)*|10011>/8 + sqrt(2)*|10100>/8 - sqrt(2)*|10101>/8 + sqrt(2)*|10110>/8 - sqrt(2)*|10111>/8 + sqrt(2)*|11000>/8 - sqrt(2)*|11001>/8 + sqrt(2)*|11010>/8 - sqrt(2)*|11011>/8 + sqrt(2)*|11100>/8 - sqrt(2)*|11101>/8 + sqrt(2)*|11110>/8 - sqrt(2)*|11111>/8
Recovered s = |1111>


In [9]:
def test_4():
    """Single-bit secret string: n=1 edge case."""
    for s in [[0], [1]]:
        size = 1
        print(f"\ntest_4: secret s = {s}")

        r = berstein_vazirani(s, size)
        m = measure_partial_oneshot(r, range(size))
        s_found = _extract_s(m, size)
        print(f"Recovered s = {s_found}")

        assert Qubit(*s) == s_found


test_4()


test_4: secret s = [0]
|x>: sqrt(2)*|0>/2 + sqrt(2)*|1>/2
|y>: sqrt(2)*|0>/2 - sqrt(2)*|1>/2
|xy>: |00>/2 - |01>/2 + |10>/2 - |11>/2
Recovered s = |0>

test_4: secret s = [1]
|x>: sqrt(2)*|0>/2 + sqrt(2)*|1>/2
|y>: sqrt(2)*|0>/2 - sqrt(2)*|1>/2
|xy>: |00>/2 - |01>/2 + |10>/2 - |11>/2
Recovered s = |1>


In [10]:
def test_5():
    """fbv inner-product correctness: manual spot-checks."""
    # s = [1,1], x = [1,0] -> 1*1 XOR 1*0 = 1
    assert fbv([1, 0], [1, 1]) == 1
    # s = [1,1], x = [1,1] -> 1 XOR 1 = 0
    assert fbv([1, 1], [1, 1]) == 0
    # s = [0,0], any x -> 0
    assert fbv([1, 1], [0, 0]) == 0
    print("test_5: fbv inner-product OK")


test_5()

test_5: fbv inner-product OK
